Build a nano GPT
-----
[video](https://youtu.be/kCc8FmEb1nY)

Initial Setup
----

1. download sample dataset
2. read the entire contenct of downloaded as text string
3. find the voca from text string
4. encode the entire text to equivalent integer ids
5. Build train and validation split from encoded data
6. get random batch of input and target data from train/validation split

In [2]:
import torch

In [3]:
# download the sample dataset(tiny shakespeare dataset)
!curl https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -o input.txt -s

In [4]:
# read the input file
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

print("total chars in text: ", len(text))

total chars in text:  1115394


In [5]:
# construct vocab
chars = sorted (list(set(text))) # extract all unique chars from entire text
vocab_size = len(chars)
print('all chars',''.join(chars))
print(f'{vocab_size=}')

all chars 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab_size=65


In [6]:
# construct encoder and decoder
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[ch] for ch in s]
decode = lambda ix: [itos[i] for i in ix]

In [7]:
# tokenization/encode the entire text to integer ids
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [8]:
## Build training & validation split from encoded data ##
n = int(0.9 * len(data)) # first 90% will be used for train and remainin 10% for validation
train_data = data[:n]
val_data = data[n:]

In [9]:
# optional - sample showing how we construct chunks of data that we later use for training
ctx_len = 8 #context length: how many characters do we take to predict the next one?
x = train_data[:ctx_len]
y = train_data[1:ctx_len+1]
for t in range(ctx_len):
  ctx = x[:t+1]
  target = y[t]
  print(f'{ctx=}, {target=}')


ctx=tensor([18]), target=tensor(47)
ctx=tensor([18, 47]), target=tensor(56)
ctx=tensor([18, 47, 56]), target=tensor(57)
ctx=tensor([18, 47, 56, 57]), target=tensor(58)
ctx=tensor([18, 47, 56, 57, 58]), target=tensor(1)
ctx=tensor([18, 47, 56, 57, 58,  1]), target=tensor(15)
ctx=tensor([18, 47, 56, 57, 58,  1, 15]), target=tensor(47)
ctx=tensor([18, 47, 56, 57, 58,  1, 15, 47]), target=tensor(58)


In [14]:
torch.manual_seed(1337) # ensures that the random number generation is reproducible,
ctx_len = 8 # context length: how many characters do we take to predict the next one
batch_size = 4 # how many independent sequences will we process in parallel

def get_batch(split): # generates batch(batch_size) of inputs and target data. Each item in batch is of size ctx_len
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - ctx_len, (batch_size,)) # generates batch_size (4) random integers.
  # generate input and output sequence based on above 4 random integers
  x = torch.stack([data[i:i+ctx_len] for i in ix]) # rows of input sequence
  y = torch.stack([data[i+1:i+ctx_len+1] for i in ix]) # rows of output sequence
  return x, y

xb, yb = get_batch('train') # get a batch of input and target
print(f'{xb=}')
print(f'{yb=}')

# just showing the chunks/grouping generated for the 1st item in batch
for t in range(ctx_len):
  ctx = xb[0, :t+1]
  target = yb[0, t]
  print(f"when input is {ctx.tolist()} the target: {target}")

xb=tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
yb=tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39


Pytorch Bigram model
-----

In [19]:
# Bigram Model Definition
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # creates a lookup table that maps each input character to a vector of scores/weights (logits) representing how likely each character is to follow it.
        # The scores are randomly initialized at first and are learned during training.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # shape of input idx=[4,8] and target=[4,8]
        # lookup against the input to get the corresponding logits
        logits = self.token_embedding_table(idx) # (B-Batch,T-Times,C-Channel)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # reshaping to conform with pytorch cross_entropy. i.e [4,8,65] to [32,65]
            targets = targets.view(B*T) # reshaping to conform with pytorch cross_entropy. i.e [4,8] to [32]
            loss = F.cross_entropy(logits, targets)

        return logits, loss

# Create a Bigram model
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)
